# Project 54 — Local Filesystem Agent

**Search, summarize, and organize files with human approval — using LangGraph.**

| Component | Choice |
|-----------|--------|
| LLM runtime | Ollama (qwen3:8b) |
| Framework | LangGraph |
| Tools | Python filesystem (pathlib, os) |
| Interface | Jupyter notebook |

## 1 · What You Will Learn

1. How to build a **stateful agent** with LangGraph that manages filesystem operations
2. How to implement a **human-in-the-loop approval** step before destructive actions
3. How to let an LLM **search and summarize** file contents
4. How to propose **file organization** plans and execute them safely
5. How to design **tool-using agents** with safety guardrails

## 2 · Why This Project Matters

Filesystem agents can automate tedious file management tasks — finding documents,
summarizing folders, organizing by topic. The key challenge is **safety**: you never
want an LLM to delete or move files without explicit human approval. This project
teaches the human-in-the-loop pattern that's critical for production agents.

## 3 · Environment Setup

**Prerequisites:**
- Ollama running with `qwen3:8b`
- Python packages below

In [ ]:
# !pip install -q langchain langchain-ollama langgraph

## 4 · Imports and Configuration

In [ ]:
import os
import json
import shutil
from pathlib import Path
from typing import TypedDict

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL = "qwen3:8b"
llm = ChatOllama(model=MODEL, temperature=0.1)

# All operations happen inside this sandbox
SANDBOX = Path('sandbox_workspace')
SANDBOX.mkdir(exist_ok=True)
print(f'LLM ready: {MODEL}')
print(f'Sandbox: {SANDBOX.resolve()}')

## 5 · Create Sample Filesystem

We create a messy folder structure to organize. This simulates a real user's
Downloads folder or project directory.

In [ ]:
# Clean and recreate sandbox
if SANDBOX.exists():
    shutil.rmtree(SANDBOX)
SANDBOX.mkdir()

# Create sample files with realistic content
sample_files = {
    'meeting_notes_jan.txt': (
        'Meeting Notes - January 2024\n'
        'Attendees: Alice, Bob, Carol\n'
        'Agenda: Q4 review, Q1 planning\n'
        'Action items:\n- Alice: finalize budget\n- Bob: hire 2 engineers'
    ),
    'meeting_notes_feb.txt': (
        'Meeting Notes - February 2024\n'
        'Attendees: Alice, David\n'
        'Agenda: Product roadmap\n'
        'Action items:\n- Launch feature X by March'
    ),
    'project_proposal.txt': (
        'Project Proposal: AI Assistant\n'
        'Budget: $50,000\nTimeline: 3 months\n'
        'Team: 2 engineers, 1 designer\nGoal: Internal productivity tool'
    ),
    'invoice_001.txt': 'Invoice #001\nClient: Acme Corp\nAmount: $15,000\nDate: 2024-01-15\nStatus: Paid',
    'invoice_002.txt': 'Invoice #002\nClient: TechStart LLC\nAmount: $8,500\nDate: 2024-02-20\nStatus: Pending',
    'random_notes.txt': 'Todo:\n- Buy groceries\n- Call dentist\n- Review PR #42\n- Update dependencies',
    'quarterly_report.txt': (
        'Q4 2023 Report\nRevenue: $1.2M\nExpenses: $900K\nProfit: $300K\n'
        'Headcount: 25\nKey wins: Launched product v2, signed 3 enterprise deals'
    ),
    'ideas.txt': 'Product Ideas:\n1. AI-powered search\n2. Auto-categorization\n3. Smart notifications',
}

for filename, content in sample_files.items():
    (SANDBOX / filename).write_text(content, encoding='utf-8')

print(f'Created {len(sample_files)} sample files in {SANDBOX}:')
for f in sorted(SANDBOX.iterdir()):
    print(f'  {f.name} ({f.stat().st_size} bytes)')

## 6 · Define Filesystem Tools

We define read-only tools (search, list, read) and write tools (move, rename)
that require approval.

In [ ]:
def list_files(directory: str = '') -> str:
    target = SANDBOX / directory
    if not target.exists():
        return f'Directory not found: {directory}'
    files = []
    for f in sorted(target.iterdir()):
        kind = 'DIR' if f.is_dir() else 'FILE'
        size = f.stat().st_size if f.is_file() else 0
        files.append(f'{kind}: {f.name} ({size} bytes)')
    return '\n'.join(files) if files else 'Empty directory'


def read_file(filename: str) -> str:
    path = SANDBOX / filename
    if not path.exists():
        return f'File not found: {filename}'
    if not str(path.resolve()).startswith(str(SANDBOX.resolve())):
        return 'Access denied: path escapes sandbox'
    return path.read_text(encoding='utf-8')


def search_files(keyword: str) -> str:
    results = []
    for f in SANDBOX.rglob('*'):
        if f.is_file():
            try:
                content = f.read_text(encoding='utf-8')
                if keyword.lower() in content.lower():
                    for line in content.split('\n'):
                        if keyword.lower() in line.lower():
                            rel = f.relative_to(SANDBOX)
                            results.append(f'{rel}: {line.strip()}')
                            break
            except Exception:
                pass
    return '\n'.join(results) if results else f'No files contain "{keyword}"'


# Test tools
print('=== File listing ===')
print(list_files())
print('\n=== Search for "invoice" ===')
print(search_files('invoice'))

## 7 · Build the Agent State and Workflow

We use LangGraph to create a stateful workflow:

```
Analyze Request → Plan Actions → [Approval Gate] → Execute → Report
```

The approval gate pauses the workflow for human review before any write operations.

In [ ]:
class AgentState(TypedDict):
    request: str
    file_listing: str
    analysis: str
    plan: str
    approved: bool
    result: str


def analyze_request(state: AgentState) -> AgentState:
    file_listing = list_files()
    analysis_prompt = ChatPromptTemplate.from_messages([
        ('system',
         'You are a filesystem assistant. Given a user request and current files, '
         'analyze what needs to be done. List specific files and actions. Be concise.'),
        ('human', 'Request: {request}\n\nCurrent files:\n{files}'),
    ])
    chain = analysis_prompt | llm | StrOutputParser()
    analysis = chain.invoke({'request': state['request'], 'files': file_listing})
    return {**state, 'file_listing': file_listing, 'analysis': analysis}


def create_plan(state: AgentState) -> AgentState:
    plan_prompt = ChatPromptTemplate.from_messages([
        ('system',
         'You are a filesystem organizer. Based on the analysis, create a concrete plan.\n'
         'Format each step as: ACTION: [details]\n'
         'Valid actions: CREATE_FOLDER, MOVE_FILE, SUMMARIZE, SEARCH\n'
         'Be specific with file names and destinations.'),
        ('human', 'Analysis:\n{analysis}\n\nFiles:\n{files}'),
    ])
    chain = plan_prompt | llm | StrOutputParser()
    plan = chain.invoke({'analysis': state['analysis'], 'files': state['file_listing']})
    return {**state, 'plan': plan}


def execute_plan(state: AgentState) -> AgentState:
    if not state.get('approved', False):
        return {**state, 'result': 'Plan was NOT approved. No changes made.'}

    results = []
    plan_lines = state['plan'].split('\n')
    for line in plan_lines:
        line = line.strip()
        if 'CREATE_FOLDER' in line.upper():
            parts = line.split(':')
            if len(parts) >= 2:
                folder_name = parts[-1].strip().strip('`').split('/')[-1]
                folder_name = ''.join(c for c in folder_name if c.isalnum() or c in '-_ ')
                if folder_name:
                    (SANDBOX / folder_name).mkdir(exist_ok=True)
                    results.append(f'Created folder: {folder_name}')
        elif 'MOVE_FILE' in line.upper():
            results.append(f'Move noted (dry-run): {line}')
        elif 'SUMMARIZE' in line.upper():
            results.append(f'Summary task noted: {line}')

    if not results:
        results.append('Plan parsed but no concrete actions extracted.')

    return {**state, 'result': '\n'.join(results)}


print('Agent functions defined.')

## 8 · Build the LangGraph Workflow

Now we wire the functions into a LangGraph graph.

In [ ]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)
workflow.add_node('analyze', analyze_request)
workflow.add_node('plan', create_plan)
workflow.add_node('execute', execute_plan)

workflow.set_entry_point('analyze')
workflow.add_edge('analyze', 'plan')
workflow.add_edge('plan', 'execute')
workflow.add_edge('execute', END)

app = workflow.compile()
print('LangGraph workflow compiled.')
print('Nodes: analyze -> plan -> execute -> END')

## 9 · Run the Agent

Let's run the agent on a realistic task: organize files by category.

In [ ]:
# Task 1: Organize files
state = app.invoke({
    'request': 'Organize these files into folders by category (meetings, finance, ideas, misc)',
    'file_listing': '', 'analysis': '', 'plan': '',
    'approved': True,  # Auto-approve for demo
    'result': '',
})

print('ANALYSIS:')
print(state['analysis'])
print('\nPLAN:')
print(state['plan'])
print('\nRESULT:')
print(state['result'])

In [ ]:
# Task 2: Search and summarize
state2 = app.invoke({
    'request': 'Find all files related to invoices and summarize the total amounts',
    'file_listing': '', 'analysis': '', 'plan': '',
    'approved': True, 'result': '',
})

print('ANALYSIS:')
print(state2['analysis'])
print('\nPLAN:')
print(state2['plan'])

## 10 · Demonstrate Approval Gate

Here we show what happens when approval is denied.

In [ ]:
state3 = app.invoke({
    'request': 'Delete all temporary files',
    'file_listing': '', 'analysis': '', 'plan': '',
    'approved': False,  # Human says NO
    'result': '',
})

print('RESULT (denied):')
print(state3['result'])

## 11 · File Summarization

Let's test the LLM's ability to read and summarize file contents.

In [ ]:
def summarize_file(filename: str) -> str:
    content = read_file(filename)
    if content.startswith('File not found') or content.startswith('Access denied'):
        return content
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'Summarize this file in 2-3 sentences. Note any action items or key numbers.'),
        ('human', 'Filename: {filename}\n\nContent:\n{content}'),
    ])
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({'filename': filename, 'content': content})


# Summarize each file
for f in sorted(SANDBOX.iterdir()):
    if f.is_file():
        print(f'\n--- {f.name} ---')
        print(summarize_file(f.name))

## 12 · Save Results

In [ ]:
output = {
    'task1_organize': {'analysis': state['analysis'], 'plan': state['plan'], 'result': state['result']},
    'task2_search': {'analysis': state2['analysis'], 'plan': state2['plan']},
    'task3_denied': {'result': state3['result']},
}
with open('filesystem_agent_results.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print('Results saved.')

## 13 · Limitations

- **Sandbox only** — all operations are restricted to the sandbox directory
- **Simulated moves** — file moves are dry-run only for safety
- **Basic plan parsing** — structured output would be more reliable
- **No undo** — production systems need reversible operations
- Human approval is simulated; real systems need interactive input

## 14 · How to Improve

1. **Use LangGraph interrupt** for real human-in-the-loop approval
2. **Add undo/rollback** by logging all operations
3. **Support binary files** with metadata extraction (PDFs, images)
4. **Add duplicate detection** by comparing file hashes
5. **Integrate with cloud storage** (S3, GCS) with the same approval pattern

## 15 · Mini Challenge

1. Add a `rename_file` tool that proposes better names based on file content
2. Add a `find_duplicates` tool that identifies files with similar content
3. Implement real file moves (not dry-run) but only after explicit approval

## 16 · Key Takeaways

| Concept | What You Practiced |
|---------|-------------------|
| LangGraph workflow | Stateful agent with typed state dict |
| Human-in-the-loop | Approval gate before destructive actions |
| Filesystem tools | Search, list, read, summarize files |
| Safety guardrails | Sandbox restriction + dry-run execution |
| Local-first | Ollama + local filesystem — no cloud needed |